---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1">
        <h2>LABAP — Laboratório de Análise de Bacias e Petrofísica</h2>
        <h3>Sonic Log Prediction — PhD Thesis</h3>
        <h1>Notebook 16 — Well Petrophysical Atlas</h1>
        <p>Visual reference of petrophysical logs, predicted DT, and absolute error
        for all wells in the LOWO benchmark.</p>
    </div>
</div>

---

## 1. Setup and Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sonic_ml_utils import LITHO_COLORS, FALLBACK_COLOR, plot_petrophysical_diagnosis_panel
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH        = Path('../data/processed/wells_iqr_with_lithology.csv')
PRED_PATH        = Path('../results/xgboost/predictions/lowo_xgboost_predictions.csv')
METRICS_PATH     = Path('../results/xgboost/metrics/lowo_xgboost_results.csv')
FORMATIONS_PATH  = Path('../data/formations.csv')

FIGURES_PATH = Path('../results/atlas/figures/')
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'XGBoost'

# ── Formation color palette ───────────────────────────────────────────────────
FORMATION_COLORS = {
    'FM. CALUMBI'       : '#2980B9',
    'FM. COTINGUIBA'    : '#27AE60',
    'FM. RIACHUELO'     : '#E67E22',
    'FM. MURIBECA'      : '#8E44AD',
    'FM. COQUEIRO SECO' : '#E74C3C',
    'out_of_range'      : '#BDC3C7',
}

# ── Helper: lithology color ───────────────────────────────────────────────────
def lith_color(lith):
    if pd.isna(lith): return FALLBACK_COLOR
    return LITHO_COLORS.get(str(lith).lower().strip(), FALLBACK_COLOR)

print('Paths and constants OK')

## 2. Data Loading and Preparation

Loads the main dataset, LOWO predictions, per-well metrics and formation intervals.
Merges all sources into a single DataFrame `df` used throughout the notebook.

In [ ]:
df_main = pd.read_csv(DATA_PATH)
df_pred = pd.read_csv(PRED_PATH)
df_metr = pd.read_csv(METRICS_PATH)
df_form = pd.read_csv(FORMATIONS_PATH, sep=';')

print(f'Main dataset   : {len(df_main):,} samples | {df_main["Well_Name"].nunique()} wells')
print(f'Predictions    : {len(df_pred):,} samples | {df_pred["Well_Name"].nunique()} wells')
print(f'LOWO metrics   : {len(df_metr)} wells')
print(f'Formation int. : {len(df_form)} intervals | {df_form["Well_Name"].nunique()} wells')
print(f'\nFormation columns: {df_form.columns.tolist()}')

In [ ]:
# ── Merge: bring Lithology + Formation into predictions ──────────────────────
cols_merge = [c for c in ['DEPTH', 'Well_Name', 'Lithology',
                           'Lithology_Segment_Thickness', 'Formation']
              if c in df_main.columns]

df = df_pred.merge(df_main[cols_merge], on=['DEPTH', 'Well_Name'], how='left')

# Bring per-well R² into df
df = df.merge(df_metr[['Well_Name', 'R2']], on='Well_Name', how='left')

# ── Derived columns ───────────────────────────────────────────────────────────
df['residual']  = df['DT_pred'] - df['DT_real']
df['abs_error'] = df['residual'].abs()

# Normalise lithology label (lowercase, strip)
df['Lithology_norm'] = df['Lithology'].str.lower().str.strip()

# Fill missing Formation
if 'Formation' not in df.columns or df['Formation'].isna().all():
    print('Assigning formations from formations.csv...')
    for _, row in df_form.iterrows():
        mask = ((df['Well_Name'] == row['Well_Name']) &
                (df['DEPTH']    >= row['depth_start']) &
                (df['DEPTH']    <  row['depth_end']))
        df.loc[mask, 'Formation'] = row['formation']
    df['Formation'] = df['Formation'].fillna('out_of_range')

print(f'Merged df: {len(df):,} rows | columns: {df.columns.tolist()}')
print(f'Wells: {df["Well_Name"].nunique()} | R² range: {df["R2"].min():.3f} – {df["R2"].max():.3f}')

## 3. Well Performance Summary

Ranked table of all wells by R² LOWO, including RMSE, MAE, Bias, dominant lithology,
and deepest formation. Provides a quick overview before the visual atlas.

In [ ]:
# ── Per-well summary ─────────────────────────────────────────────────────────
summary_rows = []
for well, grp in df.groupby('Well_Name'):
    r2   = r2_score(grp['DT_real'], grp['DT_pred'])
    rmse = np.sqrt(mean_squared_error(grp['DT_real'], grp['DT_pred']))
    mae  = mean_absolute_error(grp['DT_real'], grp['DT_pred'])
    bias = grp['residual'].mean()
    dom_lith = grp['Lithology_norm'].value_counts().idxmax() if 'Lithology_norm' in grp else '—'
    formations = grp['Formation'].dropna().unique().tolist()
    formations = [f for f in formations if f != 'out_of_range']
    deep_form  = (df_form[df_form['Well_Name'] == well]
                  .sort_values('depth_end', ascending=False)
                  ['formation'].iloc[0]
                  if len(df_form[df_form['Well_Name'] == well]) > 0 else '—')
    summary_rows.append({
        'Well'             : well,
        'R²'               : round(r2,   3),
        'RMSE (µs/ft)'     : round(rmse, 2),
        'MAE (µs/ft)'      : round(mae,  2),
        'Bias (µs/ft)'     : round(bias, 2),
        'N samples'        : len(grp),
        'Dom. Lithology'   : dom_lith,
        'Deepest Formation': deep_form,
    })

df_summary = (pd.DataFrame(summary_rows)
              .sort_values('R²')
              .reset_index(drop=True))

# Flag poor-performing wells
POOR_THRESHOLD = 0.70
df_summary['Performance'] = df_summary['R²'].apply(
    lambda r: '❌ Poor'    if r < POOR_THRESHOLD else
              '⚠️  Moderate' if r < 0.75 else '✅ Good'
)

print(f'{MODEL_NAME} — LOWO Performance Summary ({len(df_summary)} wells)\n')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(df_summary.to_string(index=False))

## 4. Petrophysical Log Atlas

Six-panel diagnostic figure per well:
**Formation column | GR | Density (RHOB) | Neutron Porosity (NPHI) |
Measured vs Predicted DT | Absolute Error by Lithology**

Wells are displayed in ascending R² order (worst-performing first).

In [ ]:
def plot_well_petrophysical_log(well_name, df, df_form,
                                  depth_col='DEPTH',
                                  lith_col='Lithology_norm',
                                  error_col='abs_error',
                                  save=True):
    """
    Six-panel diagnostic figure per well:
    Formation | GR | RHOB | NPHI | Measured vs Predicted DT | Absolute Error by Lithology

    Parameters
    ----------
    well_name : str
    df        : DataFrame — merged predictions + lithology + formation + R²
    df_form   : DataFrame — formation intervals (Well_Name, depth_start, depth_end, formation)
    depth_col : str
    lith_col  : str
    error_col : str
    save      : bool — save figure to FIGURES_PATH
    """
    sub = df[df['Well_Name'] == well_name].sort_values(depth_col).copy()
    if sub.empty:
        print(f'Well {well_name} not found.')
        return

    form_intervals = df_form[df_form['Well_Name'] == well_name].sort_values('depth_start')
    r2_well   = sub['R2'].iloc[0]
    depth_min = sub[depth_col].min()
    depth_max = sub[depth_col].max()

    fig, axes = plt.subplots(
        1, 6, figsize=(18, 14),
        sharey=True,
        gridspec_kw={'width_ratios': [0.4, 1, 1, 1, 1.5, 2.5], 'wspace': 0.06}
    )
    ax_form, ax_gr, ax_rhob, ax_nphi, ax_dt, ax_err = axes

    # ── Helper: formation background bands ────────────────────────────────────
    def add_form_background(ax):
        for _, row in form_intervals.iterrows():
            top  = max(row['depth_start'], depth_min)
            base = min(row['depth_end'],   depth_max)
            if base <= top: continue
            color = FORMATION_COLORS.get(row['formation'], '#BDC3C7')
            ax.axhspan(top, base, color=color, alpha=0.07, zorder=0)
            ax.axhline(top, color=color, lw=0.6, linestyle='--', alpha=0.4)

    # ── Panel 1: Stratigraphic column ─────────────────────────────────────────
    for _, row in form_intervals.iterrows():
        top  = max(row['depth_start'], depth_min)
        base = min(row['depth_end'],   depth_max)
        if base <= top: continue
        color = FORMATION_COLORS.get(row['formation'], '#BDC3C7')
        ax_form.barh(y=(top + base) / 2, width=1, height=(base - top),
                     color=color, edgecolor='white', linewidth=0.3)
        if (base - top) > 50:
            ax_form.text(0.5, (top + base) / 2,
                         row['formation'].replace('FM. ', ''),
                         ha='center', va='center', fontsize=5.5,
                         rotation=90, color='white', fontweight='bold')

    ax_form.set_xlim(0, 1)
    ax_form.set_xticks([])
    ax_form.set_ylabel('Depth [m]', fontsize=11)
    ax_form.set_title('Fm.', fontsize=8)
    form_handles = [
        mpatches.Patch(color=FORMATION_COLORS.get(r['formation'], '#BDC3C7'),
                       label=r['formation'].replace('FM. ', ''))
        for _, r in form_intervals.iterrows()
        if r['depth_end'] >= depth_min and r['depth_start'] <= depth_max
    ]
    ax_form.legend(handles=form_handles, loc='lower left',
                   fontsize=5.5, framealpha=0.9,
                   title='Formation', title_fontsize=6)

    # ── Panel 2: GR ───────────────────────────────────────────────────────────
    if 'GR' in sub.columns:
        add_form_background(ax_gr)
        ax_gr.plot(sub['GR'], sub[depth_col], color='#5D6D7E', lw=0.8, alpha=0.85)
        ax_gr.set_xlabel('GR\n[API]', fontsize=9)
        ax_gr.set_title('Gamma Ray', fontsize=9)
        ax_gr.set_xlim(0, 200)
        ax_gr.grid(alpha=0.25)
    else:
        ax_gr.text(0.5, 0.5, 'GR\nnot available',
                   ha='center', va='center', transform=ax_gr.transAxes)

    # ── Panel 3: RHOB ─────────────────────────────────────────────────────────
    if 'RHOB' in sub.columns:
        add_form_background(ax_rhob)
        ax_rhob.plot(sub['RHOB'], sub[depth_col], color='#8B0000', lw=0.8, alpha=0.85)
        ax_rhob.set_xlabel('RHOB\n[g/cm³]', fontsize=9)
        ax_rhob.set_title('Density', fontsize=9)
        ax_rhob.set_xlim(1.5, 3.0)
        ax_rhob.grid(alpha=0.25)
    else:
        ax_rhob.text(0.5, 0.5, 'RHOB\nnot available',
                     ha='center', va='center', transform=ax_rhob.transAxes)

    # ── Panel 4: NPHI ─────────────────────────────────────────────────────────
    if 'NPHI' in sub.columns:
        add_form_background(ax_nphi)
        ax_nphi.plot(sub['NPHI'], sub[depth_col], color='#2471A3', lw=0.8, alpha=0.85)
        ax_nphi.set_xlabel('NPHI\n[pu]', fontsize=9)
        ax_nphi.set_title('Neutron Porosity', fontsize=9)
        ax_nphi.set_xlim(80, 0)   # reversed axis
        ax_nphi.grid(alpha=0.25)
    else:
        ax_nphi.text(0.5, 0.5, 'NPHI\nnot available',
                     ha='center', va='center', transform=ax_nphi.transAxes)

    # ── Panel 5: Measured vs Predicted DT ─────────────────────────────────────
    if 'DT_pred' in sub.columns or 'DT_real' in sub.columns:
        add_form_background(ax_dt)
        if 'DT_real' in sub.columns:
            ax_dt.plot(sub['DT_real'], sub[depth_col],
                       color='black', lw=0.9, alpha=0.85, label='Measured DT')
        if 'DT_pred' in sub.columns:
            ax_dt.plot(sub['DT_pred'], sub[depth_col],
                       color='crimson', lw=0.9, alpha=0.75,
                       linestyle='--', label='Predicted DT')
        ax_dt.set_xlabel('DT\n[µs/ft]', fontsize=9)
        ax_dt.set_title('Measured vs Predicted DT', fontsize=9)
        ax_dt.legend(fontsize=7, loc='lower right', framealpha=0.85)
        ax_dt.grid(alpha=0.25)
    else:
        ax_dt.text(0.5, 0.5, 'DT\nnot available',
                   ha='center', va='center', transform=ax_dt.transAxes)

    # ── Panel 6: Absolute Error by Lithology ──────────────────────────────────
    add_form_background(ax_err)
    for lith in sub[lith_col].dropna().unique():
        s = sub[sub[lith_col] == lith]
        ax_err.scatter(s[error_col], s[depth_col],
                       color=lith_color(lith), alpha=0.4, s=6,
                       label=str(lith).replace('_', ' ').capitalize(), zorder=2)

    sub['_depth_bin'] = (sub[depth_col] // 100 * 100).astype(int)
    trend = (sub.groupby('_depth_bin')[error_col]
             .mean().reset_index()
             .rename(columns={error_col: 'mae_mean'}))
    ax_err.plot(trend['mae_mean'], trend['_depth_bin'],
                color='black', lw=2, linestyle='--',
                label='Mean per bin (100m)', zorder=5)

    ax_err.set_ylim(depth_max + 20, depth_min - 20)
    ax_err.set_xlabel('Absolute Error [µs/ft]', fontsize=10)
    ax_err.set_title('Error by Lithology', fontsize=9)
    ax_err.legend(fontsize=7, loc='lower right', framealpha=0.85)
    ax_err.grid(alpha=0.25)

    fig.suptitle(
        f'{well_name}\n'
        f'Petrophysical Logs + Measured vs Predicted DT + Error  (R²={r2_well:.3f})',
        fontsize=11, fontweight='bold', y=1
    )
    plt.tight_layout()

    if save:
        out = FIGURES_PATH / f'atlas_{well_name}.png'
        plt.savefig(out, dpi=150, bbox_inches='tight')
        print(f'Saved: {out.name}')

    plt.show()
    plt.close(fig)

print('plot_well_petrophysical_log() defined.')

In [ ]:
# ── Generate atlas figures — all wells sorted by R² (worst first) ─────────────
wells_sorted = df_metr.sort_values('R2')['Well_Name'].tolist()
print(f'Generating petrophysical log atlas for {len(wells_sorted)} wells...\n')

for i, well in enumerate(wells_sorted, 1):
    r2 = df_metr.loc[df_metr['Well_Name'] == well, 'R2'].values[0]
    print(f'[{i:02d}/{len(wells_sorted)}] {well}  R²={r2:.3f}')
    plot_well_petrophysical_log(well, df, df_form)

print('\nAtlas complete.')

## 5. Petrophysical Diagnosis Panel

Four-panel petrophysical diagnosis for each well using `plot_petrophysical_diagnosis_panel`
from `sonic_ml_utils`:

1. **RHOB × NPHI** — petrophysical space colored by absolute error,
   with P5/P95 training bounds.
2. **Measured DT × NPHI** — velocity-porosity relationship by lithology.
3. **Measured DT × Predicted DT** — 1:1 scatter with trend and statistics.
4. **Absolute Error × Depth** — error profile colored by RHOB, with Spearman correlations.

Wells are displayed in the same ascending R² order.

In [ ]:
# ── Generate diagnosis panels — all wells sorted by R² (worst first) ─────────
print(f'Generating petrophysical diagnosis panels for {len(wells_sorted)} wells...\n')

for i, well in enumerate(wells_sorted, 1):
    r2 = df_metr.loc[df_metr['Well_Name'] == well, 'R2'].values[0]
    print(f'[{i:02d}/{len(wells_sorted)}] {well}  R²={r2:.3f}')

    save_path = str(FIGURES_PATH / f'diagnosis_{well}.png')
    plot_petrophysical_diagnosis_panel(
        well_name=well,
        df_pred=df,
        save_path=save_path,
        lang='en'
    )

print('\nDiagnosis panels complete.')